# Notebook C: baseline học sâu (GRU, DLinear, GCN-GRU) trên Colab

**Mục tiêu:** đưa ba mô hình học sâu vào đúng cách chia dữ liệu và đúng bộ chỉ số của Notebook B, rồi so với LightGBM và Extra Trees (có và không có hiệu chỉnh trực tuyến C1) để quyết định ở cổng 13/10.

**Chạy trên Colab với GPU** (Runtime, Change runtime type, T4). Cần thư mục dự án gồm `data/processed/urbanev.parquet`, `data/processed/manifest.json`, `sql/queries.sql`, `src/evalkit.py` và thư mục `report/` đã có kết quả của Notebook B. Mọi mô hình dự báo đúng một đích: nhu cầu của vùng sau 24 giờ.

## Bước 0. Cấu hình và kiểm tra bàn giao

- **Làm gì:** nối Drive (nếu dùng), đọc cấu hình, kiểm tra mã băm của file parquet.
- **Vì sao:** bảo đảm Colab dùng đúng cùng một dữ liệu với Notebook A và B.
- **Kiểm tra:** ô in `handover ok`. Đặt `QUICK = True` để chạy thử khoảng vài phút (1 seed, 1 epoch, 1 kỳ nghỉ) trước khi chạy đủ; **số trong bài chỉ lấy khi `QUICK = False`.**

In [2]:
%pip install duckdb pyarrow
from google.colab import drive; drive.mount('/content/drive')
import os, sys, json, time, hashlib, warnings, urllib.request, numpy as np, pandas as pd
warnings.filterwarnings('ignore')
PROJECT = '/content/drive/MyDrive/UrbanEV_project'                      # on Colab, e.g. '/content/drive/MyDrive/urbanev'
os.chdir(PROJECT); sys.path.insert(0, 'src'); import evalkit as ek
import torch, torch.nn as nn
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', dev)
QUICK = False                    # True: smoke test only. False: the run whose numbers go into the paper
SEEDS, EPOCHS, PATIENCE, L = (range(1), 1, 1, 144) if QUICK else (range(5), 20, 3, 144)      # L = 144 h: the longest history every row has
m = json.load(open('data/processed/manifest.json')); assert hashlib.md5(open(m['file'], 'rb').read()).hexdigest() == m['md5'], 'parquet differs from the manifest'
os.makedirs('report', exist_ok=True); print('handover ok', m['rows'], 'rows')

Mounted at /content/drive
device: cuda
handover ok 1194600 rows


## Bước 1. Cùng cách chia với Notebook B

- **Làm gì:** dựng lại `feat`, 30% vùng giữ riêng, tập huấn luyện, calib và `E` (calib, ngày đệm và test của các vùng đã thấy) đúng như Notebook B.
- **Vì sao:** để so sánh mô hình học sâu với cây trên cùng các dòng. Mô hình học sâu chỉ chạy trên vùng đã thấy; vùng chưa thấy nằm ngoài phạm vi baseline này.
- **Kiểm tra:** MAE của mốc trên test phải khớp với `report/table_rq2.csv` của Notebook B (nếu bảng đó từ lần chạy cắt mẫu thì sẽ báo lệch).

In [3]:
import duckdb
con = duckdb.connect(); con.execute(open('sql/queries.sql').read())
feat = con.execute("SELECT * FROM feat ORDER BY ts, zone").df()
zones = np.sort(feat.zone.unique()); UNSEEN = set(np.random.RandomState(0).choice(zones, int(0.3 * len(zones)), replace=False))
T1, T2 = pd.Timestamp('2022-12-15'), pd.Timestamp('2023-01-05')
seen = feat[~feat.zone.isin(UNSEEN)]
train = seen[seen.ts_y < T1].sort_values(['ts', 'zone']); calib = seen[(seen.ts >= T1) & (seen.ts_y < T2)].sort_values(['ts', 'zone'])
E = seen[seen.ts >= T1].sort_values(['ts', 'zone']).reset_index(drop=True)
is_ca, is_te = (E.ts_y < T2).to_numpy(), (E.ts >= T2).to_numpy()
yE, hol = E.y.to_numpy(), (E.holiday_y == 1).to_numpy()
RE = np.where(E.festival_y == 1, 'Spring Festival', np.where(E.rain_y == 1, 'rain', 'normal'))
wape = lambda y, p: float(100 * np.abs(y - p).sum() / max(1e-9, np.abs(y).sum()))
def table(preds, mask):
    out = {}
    for n, p in preds.items():
        r = {'MAE': float(np.abs(yE[mask] - p[mask]).mean()), 'all': wape(yE[mask], p[mask])}
        for g, k in {'normal': RE == 'normal', 'rain': RE == 'rain', 'Spring Festival': RE == 'Spring Festival', 'holiday': hol}.items():
            if (mask & k).any(): r[g] = wape(yE[mask & k], p[mask & k])
        out[n] = r
    return pd.DataFrame(out).T
ref_mae = float(np.abs(yE[is_te] - E.volume.to_numpy()[is_te]).mean())
if os.path.exists('report/table_rq2.csv'):
    b = float(pd.read_csv('report/table_rq2.csv').iloc[0].MAE); print('split check:', 'OK' if abs(ref_mae - b) < 1e-6 else 'MISMATCH (Notebook B table_rq2 comes from another run)', round(ref_mae, 4), round(b, 4))
print(len(train), len(calib), int(is_te.sum()), 'test rows;', len(seen.zone.unique()), 'seen zones')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

split check: OK 49.1035 49.1035
453936 92640 250128 test rows; 193 seen zones


## Bước 2. Tensor, đồ thị và phần chuẩn hoá

- **Làm gì:** dựng chuỗi nhu cầu theo giờ của từng vùng từ file parquet, kênh ngoại sinh (giờ, thời tiết thành phố), ma trận kề của UrbanEV (thu hẹp về các vùng đã thấy) và lớp `Prep` giữ mọi thống kê tính từ tập huấn luyện.
- **Vì sao:** chuẩn hoá theo vùng bằng thống kê của riêng tập huấn luyện để không rò rỉ; hàm mất mát nhân với độ lệch chuẩn của vùng để khớp với MAE theo kWh (vùng lớn có trọng số lớn hơn, giống WAPE).
- **Kiểm tra:** ô in số nút và bậc trung bình của đồ thị (khoảng 4 đến 5 láng giềng).

In [4]:
ev = pd.read_parquet('data/processed/urbanev.parquet')
HOURS = pd.date_range(ev.ts.min(), ev.ts.max(), freq='h'); T_ = len(HOURS)
ZS = np.sort(seen.zone.unique()); N = len(ZS)
V = ev.pivot(index='ts', columns='zone', values='volume').reindex(HOURS)[ZS].ffill().fillna(0).to_numpy('float32')      # T x N raw history
WX = ev.groupby('ts')[['T', 'U', 'rain']].mean().reindex(HOURS).ffill().bfill()
HR = HOURS.hour.to_numpy(); OFF = torch.arange(-L + 1, 1, device=dev)
ADJ = 'data/raw/adj.csv'
if not os.path.exists(ADJ):
    os.makedirs('data/raw', exist_ok=True); urllib.request.urlretrieve('https://raw.githubusercontent.com/IntelligentSystemsLab/UrbanEV/main/data/adj.csv', ADJ)
A = pd.read_csv(ADJ); A.columns = A.columns.astype(int); A.index = A.columns        # rows follow the same zone order as the columns
A = A.loc[ZS, ZS].to_numpy('float32'); A = np.maximum(A, A.T); np.fill_diagonal(A, 1.0)
A_hat = torch.tensor(A / A.sum(1, keepdims=True), device=dev)                       # row-normalised: mean over the zone and its neighbours
print('graph:', N, 'nodes, mean degree', round(float(A.sum(1).mean() - 1), 2))
hidx = lambda s: ((s - HOURS[0]).dt.total_seconds() // 3600).astype(int).to_numpy()

class Prep:
    # everything that depends on the training rows (zone scale, price scale, weather scale) is computed here from `tr` only
    def __init__(s, tr):
        g = tr.groupby('zone').y; s.mu = g.mean().reindex(ZS).fillna(0).to_numpy('float32'); s.sd = np.maximum(g.std().reindex(ZS).fillna(1).to_numpy('float32'), 0.5)
        s.cs = {c: (float(tr[c].mean()), float(tr[c].std()) + 1e-6) for c in ('e_price_y', 's_price_y', 'piles')}
        th = np.unique(hidx(tr.ts)); w = WX.iloc[th]; z = (WX - w.mean()) / (w.std() + 1e-6); z['rain'] = WX['rain']
        s.exo = torch.tensor(np.column_stack([np.sin(2 * np.pi * HR / 24), np.cos(2 * np.pi * HR / 24), z['T'], z['U'], z['rain']]), dtype=torch.float32, device=dev)   # T x 5
        s.Vs = torch.tensor((V - s.mu) / s.sd, device=dev); s.mu_t, s.sd_t = torch.tensor(s.mu, device=dev), torch.tensor(s.sd, device=dev); s.w = s.sd_t / s.sd_t.mean()
    def rows(s, d):
        ti = hidx(d.ts); ni = pd.Index(ZS).get_indexer(d.zone); assert ti.min() >= L - 1 and (ni >= 0).all()
        zc = lambda c: ((d[c] - s.cs[c][0]) / s.cs[c][1]).fillna(0).to_numpy(); dw = 2 * np.pi * d.dow_y.to_numpy() / 7
        ctx = np.column_stack([d.hour_sin, d.hour_cos, np.sin(dw), np.cos(dw), d.holiday_y, zc('e_price_y'), zc('s_price_y'), zc('piles')]).astype('float32')
        T = lambda a, dt: torch.tensor(a, dtype=dt, device=dev)
        return {'ti': T(ti, torch.long), 'ni': T(ni, torch.long), 'ctx': T(ctx, torch.float32), 'y': T(d.y.to_numpy(), torch.float32)}

graph: 193 nodes, mean degree 3.09


## Bước 3. Ba mô hình

- **Làm gì:** định nghĩa GRU, DLinear (Zeng và cộng sự, 2023, bản một bước dự báo) và GCN-GRU (một lớp tích chập đồ thị ở mỗi bước thời gian rồi GRU).
- **Vì sao:** ba họ khác nhau: mạng hồi quy, mô hình tuyến tính phân rã chuỗi và mạng không gian-thời gian. Cả ba nhận 144 giờ lịch sử của vùng, kênh ngoại sinh, lịch và giá điện của giờ đích (đã biết trước), và nhúng theo vùng.
- **Kiểm tra:** ô cuối của bước này in số tham số của từng mô hình.

In [5]:
class GRUNet(nn.Module):
    def __init__(s, n, h=64, dc=8, de=8):
        super().__init__(); s.emb = nn.Embedding(n, de); s.gru = nn.GRU(6, h, batch_first=True)
        s.head = nn.Sequential(nn.Linear(h + dc + de, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(s, x, ctx, ni): _, hN = s.gru(x); return s.head(torch.cat([hN[-1], ctx, s.emb(ni)], 1)).squeeze(1)

class DLinearNet(nn.Module):                       # trend + remainder, one linear map each over the 144-hour history
    def __init__(s, n, k=25, dc=8, de=8):
        super().__init__(); s.k = k; s.lt, s.ls = nn.Linear(L, 1), nn.Linear(L, 1); s.emb = nn.Embedding(n, de)
        for l in (s.lt, s.ls): nn.init.constant_(l.weight, 1.0 / L); nn.init.zeros_(l.bias)
        s.cx = nn.Sequential(nn.Linear(dc + de, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(s, x, ctx, ni):
        v = x[:, :, 0]; pad = (s.k - 1) // 2; vp = torch.cat([v[:, :1].repeat(1, pad), v, v[:, -1:].repeat(1, pad)], 1)
        trend = vp.unfold(1, s.k, 1).mean(-1)
        return (s.lt(trend) + s.ls(v - trend)).squeeze(1) + s.cx(torch.cat([ctx, s.emb(ni)], 1)).squeeze(1)

class GCNGRUNet(nn.Module):
    def __init__(s, A_hat, g=32, h=64, dc=8, de=8):
        super().__init__(); n = A_hat.shape[0]; s.register_buffer('A', A_hat); s.w1 = nn.Linear(6, g); s.w2 = nn.Linear(6, g, bias=False)
        s.gru = nn.GRU(g, h, batch_first=True); s.emb = nn.Embedding(n, de); s.head = nn.Sequential(nn.Linear(h + dc + de, 64), nn.ReLU(), nn.Linear(64, 1))
    def forward(s, x, ctx):                        # x: (b, L, N, 6), ctx: (b, N, 8)
        b, l, n, c = x.shape; ax = torch.einsum('nm,blmc->blnc', s.A, x)            # neighbour mean at every step
        z = torch.relu(s.w1(x) + s.w2(ax)).permute(0, 2, 1, 3).reshape(b * n, l, -1)
        _, hN = s.gru(z); hN = hN[-1].reshape(b, n, -1); e = s.emb.weight.unsqueeze(0).expand(b, -1, -1)
        return s.head(torch.cat([hN, ctx, e], -1)).squeeze(-1)                      # (b, N)

KINDS = ('GRU', 'DLinear', 'GCN-GRU')
make_net = {'GRU': lambda: GRUNet(N), 'DLinear': lambda: DLinearNet(N), 'GCN-GRU': lambda: GCNGRUNet(A_hat)}
for k in KINDS: print(k, sum(p.numel() for p in make_net[k]().parameters()), 'parameters')

GRU 20617 parameters
DLinear 2411 parameters
GCN-GRU 26025 parameters


## Bước 4. Huấn luyện và dự báo

- **Làm gì:** hàm `fit_dl` huấn luyện bằng Adam với hàm mất mát L1 có trọng số theo cỡ vùng, dừng sớm theo WAPE trên calib (kiên nhẫn 3 epoch) và khôi phục trọng số tốt nhất; hàm `predict` trả dự báo theo kWh.
- **Vì sao:** dừng sớm trên calib, không dùng test; trọng số theo cỡ vùng làm hàm mất mát khớp với chỉ số chính.
- **Kiểm tra:** ô cuối của bước này chạy một epoch ngắn để kiểm tra hình dạng tensor trước khi chạy dài.

In [6]:
def inputs(P_, ti, ni):
    idx = ti[:, None] + OFF[None, :]
    return torch.cat([P_.Vs[idx, ni[:, None]].unsqueeze(-1), P_.exo[idx]], dim=-1)           # (B, L, 6)

def gcn_inputs(P_, h):
    idx = h[:, None] + OFF[None, :]
    return torch.cat([P_.Vs[idx].unsqueeze(-1), P_.exo[idx].unsqueeze(2).expand(-1, -1, N, -1)], dim=-1)          # (b, L, N, 6)

def mats(P_, R, with_y=False):
    CT = torch.zeros(T_, N, 8, device=dev); CT[R['ti'], R['ni']] = R['ctx']; out = {'CT': CT}
    if with_y:
        YS = torch.zeros(T_, N, device=dev); YS[R['ti'], R['ni']] = (R['y'] - P_.mu_t[R['ni']]) / P_.sd_t[R['ni']]
        MS = torch.zeros(T_, N, dtype=torch.bool, device=dev); MS[R['ti'], R['ni']] = True; out.update(YS=YS, MS=MS)
    return out

@torch.no_grad()
def predict(net, P_, R, kind, bs=8192):
    net.eval()
    if kind != 'GCN-GRU':
        out = torch.empty(len(R['ti']), device=dev)
        for i in range(0, len(out), bs):
            ti, ni = R['ti'][i:i + bs], R['ni'][i:i + bs]
            out[i:i + bs] = P_.mu_t[ni] + P_.sd_t[ni] * net(inputs(P_, ti, ni), R['ctx'][i:i + bs], ni)
    else:
        CT, M, hrs = mats(P_, R)['CT'], torch.zeros(T_, N, device=dev), torch.unique(R['ti'])
        for i in range(0, len(hrs), 16):
            h = hrs[i:i + 16]; M[h] = P_.mu_t + P_.sd_t * net(gcn_inputs(P_, h), CT[h])
        out = M[R['ti'], R['ni']]
    return torch.clamp(out, min=0).cpu().numpy()

def fit_dl(kind, tr, ca, seed, lr=2e-3, bs=2048, hb=8):
    torch.manual_seed(seed); np.random.seed(seed); P_ = Prep(tr); Rt, Rc = P_.rows(tr), P_.rows(ca); yc = Rc['y'].cpu().numpy()
    net = make_net[kind]().to(dev); opt = torch.optim.Adam(net.parameters(), lr=lr); best, state, bad, t0 = 1e18, None, 0, time.perf_counter()
    if kind == 'GCN-GRU': M_ = mats(P_, Rt, with_y=True); hrs = torch.unique(Rt['ti'])
    for ep in range(EPOCHS):
        net.train()
        if kind != 'GCN-GRU':
            perm = torch.randperm(len(Rt['ti']), device=dev)
            for i in range(0, len(perm), bs):
                b = perm[i:i + bs]; ti, ni = Rt['ti'][b], Rt['ni'][b]; ys = (Rt['y'][b] - P_.mu_t[ni]) / P_.sd_t[ni]
                loss = (P_.w[ni] * (net(inputs(P_, ti, ni), Rt['ctx'][b], ni) - ys).abs()).mean(); opt.zero_grad(); loss.backward(); opt.step()
        else:
            perm = hrs[torch.randperm(len(hrs), device=dev)]
            for i in range(0, len(perm), hb):
                h = perm[i:i + hb]; o = net(gcn_inputs(P_, h), M_['CT'][h]); mk = M_['MS'][h]
                loss = (P_.w[None, :] * (o - M_['YS'][h]).abs())[mk].mean(); opt.zero_grad(); loss.backward(); opt.step()
        v = wape(yc, predict(net, P_, Rc, kind))
        if v < best - 1e-4: best, bad, state = v, 0, {k: x.detach().clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
    net.load_state_dict(state); return net, P_, {'epochs': ep + 1, 'calib_wape': best, 'train_s': time.perf_counter() - t0}

# shape and speed check: one short fit per model on a thin slice of the training rows, never used for results
for k in KINDS:
    _E = EPOCHS; EPOCHS = 1; t0 = time.perf_counter(); net, P_, info = fit_dl(k, train.iloc[::20], calib.iloc[::20], 0); p = predict(net, P_, P_.rows(E.iloc[:2000]), k)
    EPOCHS = _E; print(k, 'ok', p.shape, 'calib WAPE %.1f' % info['calib_wape'], '%.1f s' % (time.perf_counter() - t0))

GRU ok (2000,) calib WAPE 29.5 13.5 s
DLinear ok (2000,) calib WAPE 25.9 0.1 s
GCN-GRU ok (2000,) calib WAPE 24.3 7.1 s


## Bước 5. Kết quả giai đoạn chính (năm seed)

- **Làm gì:** với mỗi mô hình và mỗi seed, huấn luyện trên tập huấn luyện, dừng sớm theo calib, dự báo mọi dòng của `E`, rồi chấm test hai lần: dự báo thô và dự báo qua hiệu chỉnh trực tuyến C1 (`online_correct`, mô hình là thành phần duy nhất, không trộn mốc).
- **Vì sao:** cho phép so sánh công bằng với cây ở cả hai cột; cùng tham số hiệu chỉnh nguyên bản.
- **Kiểm tra:** `report/table_dl.csv` (trung bình, độ lệch chuẩn), `report/table_dl_per_seed.csv`, `report/pred_dl_test.parquet`.

In [7]:
rs, PE = [], {}
for kind in KINDS:
    for seed in SEEDS:
        net, P_, info = fit_dl(kind, train, calib, seed); p = predict(net, P_, P_.rows(E), kind); pc = ek.online_correct(E, {'DL': p}, {'DL': 1.0})
        PE[(kind, seed)] = (p, pc)
        for name, f in {kind: p, kind + ' + correction': pc}.items(): rs.append({'method': name, 'seed': seed, **table({name: f}, is_te).iloc[0].to_dict(), **info})
        print(kind, seed, 'epochs', info['epochs'], 'calib WAPE %.2f' % info['calib_wape'], 'test WAPE %.2f / +corr %.2f' % (wape(yE[is_te], p[is_te]), wape(yE[is_te], pc[is_te])), '%.0f s' % info['train_s'])
rs = pd.DataFrame(rs); rs.to_csv('report/table_dl_per_seed.csv', index=False)
cols = ['MAE', 'all', 'normal', 'rain', 'Spring Festival']; agg = rs.groupby('method', sort=False)[cols].agg(['mean', 'std']); agg.columns = ['%s_%s' % c for c in agg.columns]
agg.to_csv('report/table_dl.csv'); print(agg.round(2).to_string())
out = E.loc[is_te, ['zone', 'ts', 'ts_y', 'y']].assign(regime=RE[is_te])
for kind in KINDS:
    out[kind] = np.mean([PE[(kind, s)][0] for s in SEEDS], axis=0)[is_te].astype('float32'); out[kind + ' + correction'] = np.mean([PE[(kind, s)][1] for s in SEEDS], axis=0)[is_te].astype('float32')
out.to_parquet('report/pred_dl_test.parquet', index=False)

GRU 0 epochs 5 calib WAPE 22.77 test WAPE 27.62 / +corr 25.53 30 s
GRU 1 epochs 7 calib WAPE 22.46 test WAPE 26.17 / +corr 25.14 44 s
GRU 2 epochs 9 calib WAPE 22.37 test WAPE 26.33 / +corr 25.04 59 s
GRU 3 epochs 5 calib WAPE 22.65 test WAPE 26.53 / +corr 25.40 33 s
GRU 4 epochs 6 calib WAPE 22.48 test WAPE 27.41 / +corr 25.42 40 s
DLinear 0 epochs 4 calib WAPE 21.69 test WAPE 24.76 / +corr 23.79 2 s
DLinear 1 epochs 6 calib WAPE 21.50 test WAPE 24.54 / +corr 23.69 3 s
DLinear 2 epochs 5 calib WAPE 21.63 test WAPE 25.63 / +corr 24.20 3 s
DLinear 3 epochs 13 calib WAPE 21.36 test WAPE 24.99 / +corr 23.87 8 s
DLinear 4 epochs 9 calib WAPE 21.50 test WAPE 24.69 / +corr 23.74 5 s
GCN-GRU 0 epochs 9 calib WAPE 22.46 test WAPE 26.89 / +corr 25.36 73 s
GCN-GRU 1 epochs 5 calib WAPE 22.85 test WAPE 26.89 / +corr 25.69 40 s
GCN-GRU 2 epochs 8 calib WAPE 22.43 test WAPE 26.65 / +corr 25.35 65 s
GCN-GRU 3 epochs 7 calib WAPE 22.82 test WAPE 27.05 / +corr 25.77 57 s
GCN-GRU 4 epochs 7 calib WAPE 

## Bước 6. Leave-one-holiday-out cho học sâu

- **Làm gì:** chạy ba mô hình trong khung LOHO của `evalkit` (một seed mỗi fold, dừng sớm trên khối calib của fold) và chấm thô cùng hiệu chỉnh.
- **Vì sao:** xem học sâu có tổng quát sang kỳ nghỉ khác ngoài Tết hay không, cùng giao thức với cây.
- **Kiểm tra:** `report/table_loho_dl.csv`. Trung Thu sát đầu dữ liệu nên tầng vùng của hiệu chỉnh gần như không chạy ở fold đó; ghi chú khi báo cáo.

In [8]:
def fit_predict_dl(tr, cx, ca, seed=0):
    out = {'Reference': cx.volume.to_numpy()}
    for kind in KINDS:
        net, P_, _ = fit_dl(kind, tr, ca, seed); p = predict(net, P_, P_.rows(cx), kind)
        out[kind] = p; out[kind + ' + correction'] = ek.online_correct(cx, {'DL': p}, {'DL': 1.0})
    return out
HOLS = {'Spring Festival': ek.HOLIDAYS['Spring Festival']} if QUICK else ek.HOLIDAYS
tl_dl = ek.run_loho(seen, fit_predict_dl, holidays=HOLS); tl_dl.to_csv('report/table_loho_dl.csv', index=False)
for mt in ('WAPE', 'MASE', 'bias_pct'): print(mt); print(tl_dl.pivot(index='method', columns='holiday', values=mt).round(2).to_string())

WAPE
holiday               Mid-Autumn  National Day  New Year  Spring Festival
method                                                                   
DLinear                    23.46         24.04     21.87            34.43
DLinear + correction       23.35         24.26     21.33            32.01
GCN-GRU                    23.91         23.13     22.64            36.74
GCN-GRU + correction       23.81         23.17     21.98            32.11
GRU                        23.23         22.93     22.05            35.94
GRU + correction           23.00         23.09     21.49            32.73
Reference                  27.42         26.04     25.22            31.14
MASE
holiday               Mid-Autumn  National Day  New Year  Spring Festival
method                                                                   
DLinear                     1.04          1.02      0.89             0.64
DLinear + correction        1.04          1.04      0.89             0.60
GCN-GRU                     

## Bước 7. Cổng quyết định 13/10

- **Làm gì:** đặt học sâu cạnh LightGBM và Extra Trees trên giai đoạn chính (thô và có hiệu chỉnh) và qua các kỳ nghỉ, rồi in kết luận theo quy tắc đã định trước.
- **Quy tắc:** học sâu **thắng** nếu WAPE tổng của mô hình học sâu tốt nhất thấp hơn LightGBM ở cả cột thô lẫn cột hiệu chỉnh với chênh lệch lớn hơn hai độ lệch chuẩn giữa các seed, **và** thắng LightGBM ở ít nhất 3 trên 4 kỳ nghỉ. Ngược lại thì hạ xuống thành baseline bổ sung.
- **Kiểm tra:** đọc cả cột `Spring Festival`, không chỉ cột tổng. Cần `report/pred_E_ml_raw.parquet` (Notebook B, ô 7.6) và `report/table_loho_ablation_wape.csv`.

In [9]:
ml = pd.read_parquet('report/pred_E_ml_raw.parquet'); assert (ml.zone.to_numpy() == E.zone.to_numpy()).all() and (ml.ts.to_numpy() == E.ts.to_numpy()).all(), 'row order differs from E'
MLP = {k: ml[k].to_numpy('float64') for k in ('Extra Trees', 'LightGBM')}
ref_rows = {'Reference (same hour today)': E.volume.to_numpy()}
for k, p in MLP.items(): ref_rows[k] = p; ref_rows[k + ' + correction'] = ek.online_correct(E, {'ML': p}, {'ML': 1.0})
tm = table(ref_rows, is_te)[['MAE', 'all', 'normal', 'rain', 'Spring Festival']]
dl_mean = agg[[c for c in agg.columns if c.endswith('_mean')]].rename(columns=lambda c: c[:-5])[['MAE', 'all', 'normal', 'rain', 'Spring Festival']]
cmp_ = pd.concat([tm, dl_mean]); cmp_.to_csv('report/table_dl_vs_ml.csv'); print(cmp_.round(2).to_string())
wins = {}
for kind in KINDS:
    for suffix, lg in (('', 'LightGBM'), (' + correction', 'LightGBM + correction')):
        sd = agg.loc[kind + suffix, 'all_std']; wins[kind + suffix] = (cmp_.loc[lg, 'all'] - cmp_.loc[kind + suffix, 'all']) > 2 * (0 if np.isnan(sd) else sd)
lo = pd.read_csv('report/table_loho_ablation_wape.csv', index_col=0); dlw = tl_dl.pivot(index='method', columns='holiday', values='WAPE')
fold_wins = {}
for kind in KINDS:
    for suffix, lg in (('', 'LightGBM (raw)'), (' + correction', 'LightGBM + correction')):
        c = [h for h in dlw.columns if h in lo.columns]; fold_wins[kind + suffix] = int((dlw.loc[kind + suffix, c] < lo.loc[lg, c]).sum())
print('beats LightGBM on the main split by > 2 sd:', wins); print('holidays won against LightGBM (of %d run):' % len(c), fold_wins)
need = 3 if len(c) == 4 else len(c)
ok = any(wins[k] and wins[k + ' + correction'] and fold_wins[k] >= need and fold_wins[k + ' + correction'] >= need for k in KINDS)
print('VERDICT:', 'deep learning wins, keep it in Table 1 as a main family' if ok else 'deep learning does not clearly beat LightGBM: move it to an additional baseline (appendix)')

                               MAE    all  normal   rain  Spring Festival
Reference (same hour today)  49.10  25.65   25.26  24.08            31.14
Extra Trees                  47.31  24.71   23.55  23.24            38.81
Extra Trees + correction     45.05  23.53   22.74  22.75            33.00
LightGBM                     48.50  25.33   24.09  24.82            39.65
LightGBM + correction        46.37  24.22   23.38  24.16            33.70
GRU                          51.33  26.81   25.58  24.11            42.58
GRU + correction             48.45  25.31   24.64  23.69            33.87
DLinear                      47.71  24.92   23.81  23.28            38.54
DLinear + correction         45.68  23.86   23.14  23.09            32.40
GCN-GRU                      51.62  26.96   25.70  24.74            42.72
GCN-GRU + correction         48.93  25.56   24.71  24.18            36.09
beats LightGBM on the main split by > 2 sd: {'GRU': np.False_, 'GRU + correction': np.False_, 'DLinear': np.Fals

In [13]:
from google.colab import files
files.download('report/table_dl_vs_ml.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>